# Coefficient cross-validation - evanescent input

Same grating, same angles, same wavelengths as `../1D_isotropic_grating_propagating_input`.
The one difference is the incident wave: `k_parallel = 1.2 > n_top`, so `sin(theta) > 1` and
the zeroth order is past the light line. meent reaches that through a **complex theta**,
`asin(1.2) = pi/2 + 0.622363i`; RETICOLO through `k_parallel = 1.2` passed straight to `res1`.

## Why this case cannot reuse the propagating recipe

**Efficiency stops being defined.** The incident wave carries no z-directed flux, so "fraction
of incident power" has a zero denominator. meent reports `R + T = 4.09` here. That is not a
solver error - the quantity has no meaning, and every efficiency-based check in
every efficiency-based check is inapplicable to this case.

**The normalization bridge inverts.** `f = Re(kz/(n_top*cos(theta)))` is what maps meent onto
RETICOLO's flux normalization. With evanescent input `cos(theta)` is purely imaginary, and `f`
becomes zero for exactly the propagating orders and positive for the evanescent ones:

```
k_par = 0.5 :  f = [0.682, 1.101, 1.000, 0,     0    ]   <- weight on propagating orders
k_par = 1.2 :  f = [0,     0,     1.000, 2.157, 3.153]   <- propagating orders zeroed
```

Multiplying by `sqrt(f)` would zero out every propagating output order. So the meent sweep
runs in `mode='raw'` and writes un-rescaled coefficients for diagnostics. They are **not**
compared directly with RETICOLO amplitudes: those use a flux-normalized modal basis, and no
common normalization exists until both fields refer to the same nonzero incident component.

**Coefficients are the only route left.** That is the point of this case - it is not reachable
by the efficiency comparison at all.

## What is already known

A prior ad-hoc Fresnel-Airy check of a uniform slab gave unit magnitude ratio and zero phase
spread at `k_parallel = 1.2` and `1.8`. The calculation is not stored here, so treat that as
an observation rather than reproducible validation evidence for now.

## Open question this case answers

RETICOLO's `amplitude` is flux-normalized and is not directly comparable with meent's raw
coefficient here. The `.m` file exports output `E` and `H` too, but those fields are still
constructed from RETICOLO's normalized modal basis. Completing this case requires an incident
E/H reference and a matching meent field-basis conversion. Whether RETICOLO accepts
`k_parallel > 1` at all is also unverified.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

CASE_DIR_NAME = '1D_isotropic_grating_evanescent_input (X)'
SEARCH_START = Path.cwd().resolve()
SIMULATION_ROOT = next((p / 'validation' / 'simulation'
                         for p in (SEARCH_START, *SEARCH_START.parents)
                         if (p / 'validation' / 'simulation' / '_compare.py').is_file()), None)
if SIMULATION_ROOT is None:
    raise RuntimeError('Cannot locate validation/simulation from the current directory')
WORKDIR = SIMULATION_ROOT / CASE_DIR_NAME
MEENT_ROOT = SIMULATION_ROOT.parents[1]
sys.path.insert(0, str(MEENT_ROOT))
sys.path.insert(0, str(SIMULATION_ROOT))

import meent
import _compare as C

print('meent   :', meent.__file__)
print('workdir :', WORKDIR)
assert str(MEENT_ROOT) in meent.__file__, (
    'meent came from elsewhere - restart the kernel so the sys.path insert wins')

In [ ]:
case = C.Case(
    name='1D_isotropic_grating_evanescent_input',
    k_parallel=1.2,        # > n_top -> complex theta, zeroth order evanescent
    phi_deg=30.0,
    # General-wavelength sweep: offset from exact modal cutoffs.
    wavelength_m=(500.5e-9, 700.5e-9, 1e-9),
    quick=False,
)
print(f'case   : {case.name}')
print(f'mode   : {case.mode}   (evanescent input: {case.evanescent})')
print(f'theta  : {case.theta.real:.6f}{case.theta.imag:+.6f}j rad   '
      f'sin(theta) = {np.sin(case.theta).real:.4f}')
print(f'fto    : {case.fto}   order window: +-{case.order_window}')
print(f'lambda : {len(case.wavelengths)} points '
      f'({case.wavelengths[0]:.6g}-{case.wavelengths[-1]:.6g} m)')

## 1. Which orders actually propagate

With the zeroth order past the light line, the propagating set is shifted. These orders carry
power out and are what RETICOLO `res2` normally exposes directly. Evanescent orders can also
be compared, but only through consistently normalized fields at a common reference plane.

In [ ]:
mee = C.make_mee(case, 'continuous', 0, float(case.wavelengths[len(case.wavelengths)//2]))
res = mee.conv_solve().res
f_r, f_t, kz_top, kz_bot = C.kz_factors(case, mee)
kz_top = kz_top.flatten()
prop = kz_top.real.abs() > 1e-12

print(f'lambda = {float(mee.wavelength):.6g} m, '
      f'{int(prop.sum())} propagating of {len(prop)} orders')
print(f'cos(theta) = {complex(torch.cos(mee.theta)):.6f}   (imaginary -> no incident z-flux)')
print(f'R + T = {float(res.de_ri.sum() + res.de_ti.sum()):.6f}   '
      f'(meaningless here - zero incident flux)\n')
print(f"{'m':>4} {'kz_top':>26} {'f':>10} {'|R_s| raw':>12} {'state':>11}")
print('-' * 68)
for m in range(-5, 4):
    i = C.order_index(case, m)
    k = kz_top[i].item()
    print(f'{m:4d} {k.real:+11.6f}{k.imag:+11.6f}j {f_r.flatten()[i].item():10.4f} '
          f'{abs(res.R_s.flatten()[i]):12.3e} '
          f'{"propagating" if prop[i] else "evanescent":>11}')

## 2. meent sweep

`mode='raw'` here, so the coefficients are written without any rescaling.

In [ ]:
failures_sweep, selfcheck = C.run_sweep(case, WORKDIR)

### Self-check

The check is run on the **flux-weighted** amplitudes even though the files contain raw ones.
`sum|r_raw*sqrt(f)|^2 == sum|r_raw|^2*f == de` is an identity; `sum|r_raw|^2` on its own is not, so
checking the raw amplitudes against `de` would report a meaningless mismatch (7.2 here) and
throw away the one thing this check is for - confirming that the `kz` branch selection in
`_compare.py` still matches the solver's.

It says nothing about physical efficiency in this case, because there is none to say anything
about.

In [ ]:
dev = C.check_selfcheck(case, selfcheck)

## 3. Magnitude, per order (blocked pending field normalization)

The shared routine deliberately skips this in `mode='raw'`. Phase-free does not mean
normalization-free: meent raw coefficients and RETICOLO flux-normalized amplitudes are not
the same quantity for an evanescent incident wave.

In [ ]:
per_order, unmatched, mag_worst = C.compare_magnitude(case, WORKDIR)

The table above is a maximum over every order at once, which is how one bad order hides behind
a good one. Split by order:

In [ ]:
C.report_per_order(case, per_order, unmatched)

## 4. Complex amplitude (blocked pending field normalization)

Once both solvers use a common physical-field normalization, one incident phase may be removed
per wavelength over every common order, both polarization components, and both directions.
That normalization is not implemented yet, so the shared routine skips instead of fitting
unrelated coefficient conventions.

Both choices are deliberate. Referencing each component to its own zeroth order would hide a
constant phase offset between TE and TM - the cross-polarization term that matters under
conical incidence. Fitting r and t separately would absorb a constant phase error applied to
every `t_m` while `r` stayed correct.

A different time convention is **not** removed this way: conjugation is not a multiplication
by a phase. It is tested as its own hypothesis and reported.

In [ ]:
print(f"{'pol':4s} {'method':11s} {'max err':>11s} {'median':>11s} {'n_wl':>6s}")
print('-' * 46)
phase_worst, conj_votes, phase_n = C.compare_phase(case, WORKDIR)

In [ ]:
N_WORST = 12
print('worst magnitude deviations')
print(f"{'d|a|':>11s} {'pol':4s} {'dir':4s} {'method':11s} {'lambda':>9s} {'order':>6s}")
print('-' * 54)
for v, pol, d, method, lam, m in sorted(mag_worst, reverse=True)[:N_WORST]:
    print(f'{v:11.3e} {C.POL_NAME[pol]:4s} {d:4s} {method:11s} {lam:11.6g} m {m:6d}')

print()
print('worst direct complex deviations')
print(f"{'err':>11s} {'pol':4s} {'method':11s} {'lambda':>9s} {'alpha (rad)':>12s}")
print('-' * 52)
for v, pol, method, lam, alpha in sorted(phase_worst, reverse=True)[:N_WORST]:
    print(f'{v:11.3e} {C.POL_NAME[pol]:4s} {method:11s} {lam:11.6g} m {alpha:12.5f}')

## 5. Verdict

This cell currently **raises intentionally**. A PASS is unavailable until RETICOLO has run and
both solvers' E/H are normalized to the same incident field. Diagnostic raw coefficient files
therefore cannot be mistaken for completed cross-validation.

In [ ]:
C.verdict(case, WORKDIR, failures_sweep, dev, per_order, unmatched, phase_worst, phase_n)

## 6. Plots

Real and imaginary parts of the complex coefficient per order, **not** `|a|`. A magnitude
plot hides what this case exists to catch: two solvers can agree on every magnitude and
still differ by a sign, a conjugation, or a per-order reference-plane phase. The verdict is
a direct complex difference, so the plot shows the same numbers it is computed from.


In [ ]:
# show_reference=False: meent raw coefficients and RETICOLO flux-normalized amplitudes
# are not on a common normalization here, so an overlay would draw two different quantities.
fig = C.plot_coefficients(case, WORKDIR, pol=0, show_reference=False)
plt.show()
